# RCM check 1 - Does the RCM reproduce the hosing response of its driving GCM?

**Ad hoc test, outside the chain**: it reads the RegCM5 0.25 deg runs of the 2050 pair (`rcm_monthly`, Base / Hose)
and the EC-Earth3 runs of the same pair (Original / Hosing), and writes only a cache of monthly means on EOS
(`outputs/rcm_check/cache/`) and figures / tables in `outputs/` next to this notebook.

**Question.** Step 2 of the plan bias-adjusts each RCM run with a QDM against the GCM product of step 1 (climate
imprint + QDM, EC-Earth3 at 0.25 deg). The QDM keeps the change simulated by the RCM, quantile by quantile: step 2 is
only worth doing if the hosing response of the RCM is credible. The ECOADAPT diagnostics (Nov 2026) show a strong
summer warm-dry bias of RegCM5 and a hosing effect on hot-summer exposure that differs between the 0.75 and the
0.25 deg configurations. Test: is the hosing minus no-hosing change of the RCM (0.25 deg) consistent with that of
EC-Earth3, at the scale of the GCM?

**Method.**
- Years `YEARS` (default 2130-2149, the same in both models): the first 5 years of the RCM runs are left out, since the
  land surface of the RCM was not spun up (ECOADAPT drift check: surface fluxes and topsoil water adjust within
  1-5 years, deep soil water within 16-18 years; 2 m temperature without trend).
- Monthly means. RCM: one file per month, the month taken from the file name (RegCM stamps day D at D+1 00:00, so the
  file of month M holds exactly the days of M). GCM: daily files (`ba_utils.gcm_ba_input`), monthly means.
- RCM aggregated onto the EC-Earth3 grid: mean of the RCM cells whose centre falls in a GCM cell, outside the lateral
  relaxation zone (`SPONGE` cells along each edge), GCM cells covered at least `MIN_COVER` by such RCM cells.
- Domain: LISFLOOD domain on the EC-Earth3 grid without the land outside Europe (`ba_utils.load_domain_mask`), within
  the RCM coverage; spatial statistics weighted by cos(lat).
- Seasonal means per year (DJF, MAM, JJA, SON, year; months weighted by their number of days; DJF of year y = Dec y-1
  + Jan, Feb y). Change = hosing - no hosing for tasmax and tasmin (K), relative change for pr (% of the no-hosing
  mean; cells with a no-hosing mean below `PR_MIN` left out).
- Statistics per variable and season: area mean of the change in each model; centred pattern correlation r, RMSD
  and ratio of the spatial standard deviations (RCM / GCM) of the two change fields; share of the area where the
  change is significant (Welch t-test on the yearly seasonal means, `ALPHA`) and sign agreement where both are
  significant.
- Noise: bootstrap of the years (`N_BOOT`, each of the four runs resampled independently) for 90 % intervals of the
  mean changes and of r (for r, the spread of the replicates around their median, centred on r: resampling the years
  lowers r in every replicate). Noise ceiling: split-half reproducibility of each model (change of the first half of `YEARS`
  against that of the second half), corrected to the full length with the Spearman-Brown formula (reliability
  rho = 2 r_half / (1 + r_half)). With the same true change in both models, internal variability alone limits the
  expected correlation to r_max = sqrt(rho_GCM rho_RCM); r / r_max is the correlation corrected for that noise.

**Reading.** For a variable and season, the RCM change is consistent with its driver when r / r_max is close to 1 and
the mean changes agree within their intervals: step 2 then adds fine-scale structure without changing the signal.
When r / r_max is clearly below 1, or the mean changes differ beyond the intervals, the RCM changes the hosing signal
and step 2 would carry that change into the product. A disagreement alone does not say which model is right; the GCM
is the model whose AMOC experiments are documented (Meccia et al. 2023; Volpi et al. seminar, Jan 2026) and whose
product (step 1) forces LISFLOOD anyway for the 2025 pair (no RCM run with hosing for 2025). Only the 2050 pair is
tested here.

In [ ]:
import os
os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import sys, glob, re, time, warnings, multiprocessing
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path
for _p in reversed((Path.cwd(), *Path.cwd().parents)):   # ba_utils.py (repository root): this folder or any parent folder
    sys.path.insert(0, str(_p))
import numpy as np, pandas as pd, xarray as xr
import matplotlib.pyplot as plt
from scipy import sparse, stats
import ba_utils as bu
warnings.filterwarnings("ignore")

CONFIG_PATH = None
cfg = bu.load_config(CONFIG_PATH)
RCM_ROOT = Path(cfg["data_root"]) / "rcm_monthly"
RCM_RUNS = {"nohos": ("stabilization-ssp585-2050_Base", "2050_Base"),   # folder, file tag
            "hos": ("stabilization-ssp585-2050_Hose", "2050_Hose")}
GCM_RUNS = {"nohos": "gcm_2050_nohosing", "hos": "gcm_2050_hosing"}   # datasets of the config
VARS = ["tasmax", "tasmin", "pr"]
DATA_YEARS = (2125, 2149)                     # years of the RCM 2050 runs; the GCM is read over the same years
YEARS = tuple(int(y) for y in os.environ.get("SIGNAL_YEARS", "2130,2149").split(","))   # years of the comparison
SPONGE = 11 + 2                               # cells left out along each RCM edge (11-cell relaxation zone + 2)
MIN_COVER = 0.5                               # GCM cells kept if covered at least this much by the RCM cells used
PR_MIN = 0.1                                  # mm/day: no relative pr change where the no-hosing mean is below
SEASONS = {"DJF": (12, 1, 2), "MAM": (3, 4, 5), "JJA": (6, 7, 8), "SON": (9, 10, 11), "year": tuple(range(1, 13))}
ALPHA = 0.05                                  # level of the t-tests
N_BOOT = 1000                                 # bootstrap replicates of the years
SEED = 20261120
N_PROC = int(os.environ.get("N_PROC", "4"))   # processes reading the RCM monthly files
OVERWRITE = False

CACHE = bu.out_dir(cfg, "rcm_check", "cache")   # the only folder written on EOS
OUT = Path.cwd() / "outputs"; OUT.mkdir(exist_ok=True)
TAG = f"{YEARS[0]}-{YEARS[1]}"
assert DATA_YEARS[0] <= YEARS[0] <= YEARS[1] <= DATA_YEARS[1]
print(f"comparison years {TAG} | cache {CACHE} | outputs {OUT}")

## 1. Grids: RCM cells used, aggregation onto the EC-Earth3 grid, domain

In [ ]:
def rcm_files(run, var):
    # monthly RCM files of DATA_YEARS, as (path, 'YYYYMM'), in time order
    folder, tag = RCM_RUNS[run]
    out = []
    for f in sorted(glob.glob(str(RCM_ROOT / folder / f"{var}_{tag}_[0-9][0-9][0-9][0-9][0-9][0-9].nc"))):
        ym = re.search(r"_(\d{6})\.nc$", f).group(1)
        if DATA_YEARS[0] <= int(ym[:4]) <= DATA_YEARS[1]:
            out.append((f, ym))
    return out


def rcm_latlon(ds):
    # 2-D latitude / longitude of the RegCM curvilinear grid
    for la, lo in (("lat", "lon"), ("xlat", "xlon"), ("latitude", "longitude")):
        if la in ds.variables and lo in ds.variables and ds[la].ndim == 2:
            return ds[la].values.astype(float), ds[lo].values.astype(float)
    raise KeyError(f"no 2-D lat / lon in the RCM file (variables: {list(ds.variables)})")


n_exp = 12 * (DATA_YEARS[1] - DATA_YEARS[0] + 1)
for run in RCM_RUNS:
    for v in VARS:
        n = len(rcm_files(run, v))
        print(f"RCM {run:5s} {v:6s}: {n} monthly files (expected {n_exp})")
        assert n == n_exp, "missing RCM months"

F0 = rcm_files("nohos", "tasmax")[0][0]
with xr.open_dataset(F0, decode_times=False) as ds0:
    RLAT, RLON = rcm_latlon(ds0)
    RDIMS = ds0["tasmax"].dims[-2:]
RLON = ((RLON + 180.0) % 360.0) - 180.0
NY, NX = RLAT.shape
INNER = np.zeros((NY, NX), bool)
INNER[SPONGE:NY - SPONGE, SPONGE:NX - SPONGE] = True
REGION = {"lat": (RLAT.min() - 1.0, RLAT.max() + 1.0), "lon": (RLON.min() - 1.0, RLON.max() + 1.0)}
print(f"RCM grid {NY} x {NX} (dims {RDIMS}), lat {RLAT.min():.1f}..{RLAT.max():.1f}, lon {RLON.min():.1f}..{RLON.max():.1f};"
      f" {INNER.sum()} cells outside the relaxation zone")

# EC-Earth3 grid over the region
G0 = bu.gcm_ba_input(cfg, GCM_RUNS["nohos"], "tasmax", (str(YEARS[0]), str(YEARS[0])), REGION).isel(time=0).load()
GLAT, GLON = G0.lat.values, G0.lon.values
NG = GLAT.size * GLON.size


def cell_area_km2(lat, lon):
    # area of the cells of a curvilinear grid from the distances to the next cell along each axis (km2)
    R = 6371.0
    la, lo = np.deg2rad(lat), np.deg2rad(lon)
    d = lambda a1, o1, a2, o2: 2 * R * np.arcsin(np.sqrt(np.sin((a2 - a1) / 2) ** 2
                                                       + np.cos(a1) * np.cos(a2) * np.sin((o2 - o1) / 2) ** 2))
    dx = d(la[:, :-1], lo[:, :-1], la[:, 1:], lo[:, 1:]); dx = np.c_[dx, dx[:, -1:]]
    dy = d(la[:-1], lo[:-1], la[1:], lo[1:]); dy = np.r_[dy, dy[-1:]]
    return dx * dy


# aggregation: RCM cell (flattened index) -> GCM cell containing its centre
ye, xe = bu._edges(GLAT), bu._edges(GLON)
iy, ix = np.searchsorted(ye, RLAT) - 1, np.searchsorted(xe, RLON) - 1
ok = INNER & (iy >= 0) & (iy < GLAT.size) & (ix >= 0) & (ix < GLON.size)
COL = np.flatnonzero(ok)
ROW = (iy * GLON.size + ix).ravel()[COL]
ONES = sparse.csr_matrix((np.ones(COL.size), (ROW, COL)), shape=(NG, NY * NX))
g_area = (np.diff(np.sin(np.deg2rad(np.clip(ye, -90, 90))))[:, None] * np.deg2rad(np.diff(xe))[None, :]) * 6371.0 ** 2
COVER = (np.bincount(ROW, weights=cell_area_km2(RLAT, RLON).ravel()[COL], minlength=NG) / g_area.ravel()).reshape(GLAT.size, GLON.size)


def to_gcm(a):
    # (..., NY, NX) RCM fields -> (..., nlat, nlon) means of the RCM cells used in each GCM cell (NaN elsewhere)
    a = np.asarray(a, float)
    flat = a.reshape(-1, NY * NX)
    fin = np.isfinite(flat)
    num = ONES @ np.where(fin, flat, 0.0).T
    den = ONES @ fin.T.astype(float)
    out = np.where(den > 0, num / np.where(den > 0, den, 1), np.nan).T
    return out.reshape(a.shape[:-2] + (GLAT.size, GLON.size))


DOM = bu.load_domain_mask(cfg, "ecearth3", like=G0).values & (COVER >= MIN_COVER)
IDX = np.flatnonzero(DOM.ravel())
W = np.broadcast_to(np.cos(np.deg2rad(GLAT))[:, None], DOM.shape).ravel()[IDX]
W = W / W.sum()
print(f"EC-Earth3 grid {GLAT.size} x {GLON.size} | GCM cells of the comparison (European LISFLOOD domain, cover >= "
      f"{MIN_COVER}): {IDX.size}")

BORDERS = bu.borders(cfg, "10M")
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].pcolormesh(GLON, GLAT, np.where(COVER > 0, np.minimum(COVER, 1.5), np.nan), shading="nearest", cmap="viridis")
ax[0].set_title("share of each GCM cell covered by the RCM cells used")
ax[1].pcolormesh(GLON, GLAT, np.where(DOM, 1.0, np.nan), shading="nearest", cmap="Greens", vmin=0, vmax=1.5)
ax[1].set_title(f"GCM cells of the comparison ({IDX.size})")
for a in ax:
    bu.draw_borders(a, BORDERS, lw=0.3); a.set_xlim(*REGION["lon"]); a.set_ylim(*REGION["lat"])
plt.tight_layout(); plt.savefig(OUT / "domain.png", dpi=110); plt.show()

## 2. Monthly means (cached on EOS)

RCM: the native monthly means (relaxation zone included, for the maps) and their aggregation onto the GCM grid. GCM:
monthly means of the daily files over the region. Units: degC and mm/day.

In [ ]:
def _rcm_month_mean(args):
    # mean of one monthly RCM file over its days, on the native grid (worker process)
    f, var = args
    with xr.open_dataset(f, decode_times=False) as ds:
        a = ds[var].squeeze(drop=True)
        tdim = [d for d in a.dims if d not in RDIMS]
        assert len(tdim) == 1, f"{f}: dimensions {a.dims}"
        return a.mean(tdim[0]).transpose(*RDIMS).values.astype("float32"), int(a.sizes[tdim[0]])


def rcm_units(var):
    # (offset, factor) from the units of the first file to degC / mm per day
    with xr.open_dataset(rcm_files("nohos", var)[0][0], decode_times=False) as ds:
        u = str(ds[var].attrs.get("units", "")).lower().replace(" ", "")
    if var == "pr":
        if u in ("kgm-2s-1", "kgm**-2s**-1", "kg/m2/s", "mm/s", "kgm^-2s^-1"):
            return 0.0, 86400.0
        if u in ("mm/d", "mm/day", "mmday-1", "mmd-1"):
            return 0.0, 1.0
    elif u in ("k", "kelvin"):
        return -273.15, 1.0
    elif u in ("degc", "c", "celsius"):
        return 0.0, 1.0
    raise ValueError(f"RCM {var}: unknown units '{u}'")


def month_index(yms):
    return pd.DatetimeIndex([pd.Timestamp(int(s[:4]), int(s[4:6]), 1) for s in yms])


def rcm_monthly(run, var):
    path = CACHE / f"rcm025_{RCM_RUNS[run][1]}_{var}_monthly_native_{DATA_YEARS[0]}-{DATA_YEARS[1]}.nc"
    if path.exists() and not OVERWRITE:
        return xr.open_dataarray(path).load()
    files = rcm_files(run, var)
    off, fac = rcm_units(var)
    t0 = time.time()
    jobs = [(f, var) for f, _ in files]
    if N_PROC > 1:
        with ProcessPoolExecutor(N_PROC, mp_context=multiprocessing.get_context("fork")) as ex:
            res = list(ex.map(_rcm_month_mean, jobs, chunksize=4))
    else:
        res = [_rcm_month_mean(j) for j in jobs]
    ndays = np.array([n for _, n in res])
    assert ndays.min() >= 28 and ndays.max() <= 31, f"{run} {var}: {ndays.min()}..{ndays.max()} days per file"
    data = (np.stack([m for m, _ in res]) + off) * fac
    da = xr.DataArray(data.astype("float32"), dims=("time", "y", "x"), name=var,
                      coords={"time": month_index([ym for _, ym in files]),
                              "lat": (("y", "x"), RLAT), "lon": (("y", "x"), RLON)},
                      attrs={"units": "mm/d" if var == "pr" else "degC", "source": f"{RCM_ROOT}/{RCM_RUNS[run][0]}"})
    tmp = path.with_suffix(".tmp.nc")
    da.to_netcdf(tmp, encoding={var: {"zlib": True, "complevel": 1}}); tmp.replace(path)
    print(f"  RCM {run} {var}: {len(files)} months in {time.time() - t0:.0f} s, {ndays.min()}-{ndays.max()} days per file")
    return da


def gcm_monthly(run, var):
    path = CACHE / f"ecearth3_{GCM_RUNS[run]}_{var}_monthly_{DATA_YEARS[0]}-{DATA_YEARS[1]}.nc"
    if path.exists() and not OVERWRITE:
        return xr.open_dataarray(path).load()
    t0 = time.time()
    d = bu.gcm_ba_input(cfg, GCM_RUNS[run], var, (str(DATA_YEARS[0]), str(DATA_YEARS[1])), REGION)
    m = d.resample(time="MS").mean().load()
    m = m.assign_coords(time=pd.DatetimeIndex([pd.Timestamp(t.year, t.month, 1) for t in m.indexes["time"]]))
    m.name = var
    tmp = path.with_suffix(".tmp.nc")
    m.to_netcdf(tmp); tmp.replace(path)
    print(f"  GCM {run} {var}: {m.sizes['time']} months in {time.time() - t0:.0f} s")
    return m


RCM_NAT, RCM, GCM = {}, {}, {}
for var in VARS:
    for run in RCM_RUNS:
        RCM_NAT[run, var] = rcm_monthly(run, var)
        RCM[run, var] = xr.DataArray(to_gcm(RCM_NAT[run, var].values), dims=("time", "lat", "lon"),
                                     coords={"time": RCM_NAT[run, var].time, "lat": GLAT, "lon": GLON})
        GCM[run, var] = gcm_monthly(run, var)
        assert bu.same_grid(GCM[run, var], G0) and GCM[run, var].sizes["time"] == n_exp
        GCM[run, var] = GCM[run, var].assign_coords(lat=GLAT, lon=GLON)
    print(f"{var}: monthly means ready")

# quick check: domain means of the no-hosing runs over YEARS
for var in VARS:
    r = float((RCM["nohos", var].sel(time=slice(str(YEARS[0]), str(YEARS[1]))).mean("time").values.ravel()[IDX] * W).sum())
    g = float((GCM["nohos", var].sel(time=slice(str(YEARS[0]), str(YEARS[1]))).mean("time").values.ravel()[IDX] * W).sum())
    print(f"{var:6s} no hosing, domain mean {TAG}: GCM {g:7.2f} | RCM {r:7.2f}")

## 3. Changes, pattern statistics, significance, noise ceiling, bootstrap

In [ ]:
DAYS = np.array([31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31], float)


def season_means(da, months, years=YEARS):
    # yearly seasonal means (n_years, ...), months weighted by their days; DJF of year y = Dec y-1 + Jan, Feb y;
    # incomplete seasons left out
    t = pd.DatetimeIndex(da["time"].values)
    djf = len(months) == 3 and 12 in months
    sy = t.year + (djf & (t.month == 12)).astype(int)
    v, out = da.values, []
    for y in range(years[0], years[1] + 1):
        k = np.flatnonzero(np.isin(t.month, months) & (sy == y))
        if len(k) == len(months):
            w = DAYS[t.month[k] - 1]
            out.append(np.tensordot(w / w.sum(), v[k], axes=1))
    return np.stack(out)


def change(hos, nohos, var):
    if var == "pr":
        return np.where(nohos >= PR_MIN, 100.0 * (hos / np.where(nohos > 0, nohos, 1.0) - 1.0), np.nan)
    return hos - nohos


def wstats(a, b, w=W):
    # area-weighted means, centred pattern correlation, RMSD and SD ratio of two change fields (NaN cells left out)
    ok = np.isfinite(a) & np.isfinite(b)
    if ok.sum() < 3:
        return dict(mean_gcm=np.nan, mean_rcm=np.nan, r=np.nan, rmsd=np.nan, sd_ratio=np.nan)
    w, a, b = w[ok] / w[ok].sum(), a[ok], b[ok]
    ma, mb = (w * a).sum(), (w * b).sum()
    sa, sb = np.sqrt((w * (a - ma) ** 2).sum()), np.sqrt((w * (b - mb) ** 2).sum())
    return dict(mean_gcm=ma, mean_rcm=mb, r=(w * (a - ma) * (b - mb)).sum() / (sa * sb),
                rmsd=np.sqrt((w * (b - a) ** 2).sum()), sd_ratio=sb / sa)


def reliability(Y, var):
    # Spearman-Brown corrected split-half reproducibility of the change field of one model
    n = Y["hos"].shape[0]; h = n // 2
    c1 = change(Y["hos"][:h].mean(0), Y["nohos"][:h].mean(0), var)
    c2 = change(Y["hos"][n - h:].mean(0), Y["nohos"][n - h:].mean(0), var)
    r = wstats(c1, c2)["r"]
    return r, (2 * r / (1 + r) if r > 0 else np.nan)


rng = np.random.default_rng(SEED)
rows, MAPS = [], {}
for var in VARS:
    for s, months in SEASONS.items():
        Yg = {run: season_means(GCM[run, var], months).reshape(-1, DOM.size)[:, IDX] for run in GCM_RUNS}
        Yr = {run: season_means(RCM[run, var], months).reshape(-1, DOM.size)[:, IDX] for run in RCM_RUNS}
        n = Yg["hos"].shape[0]
        dg = change(Yg["hos"].mean(0), Yg["nohos"].mean(0), var)
        dr = change(Yr["hos"].mean(0), Yr["nohos"].mean(0), var)
        st = wstats(dg, dr)
        sig_g = stats.ttest_ind(Yg["hos"], Yg["nohos"], axis=0, equal_var=False).pvalue < ALPHA
        sig_r = stats.ttest_ind(Yr["hos"], Yr["nohos"], axis=0, equal_var=False).pvalue < ALPHA
        both = sig_g & sig_r & np.isfinite(dg) & np.isfinite(dr)
        agree = float((W * (both & (np.sign(dg) == np.sign(dr)))).sum() / (W * both).sum()) if both.any() else np.nan
        rh_g, rel_g = reliability(Yg, var)
        rh_r, rel_r = reliability(Yr, var)
        r_max = np.sqrt(rel_g * rel_r)
        bs = []
        for _ in range(N_BOOT):
            pick = lambda Y: Y[rng.integers(0, n, n)].mean(0)
            bs.append(wstats(change(pick(Yg["hos"]), pick(Yg["nohos"]), var), change(pick(Yr["hos"]), pick(Yr["nohos"]), var)))
        bs = pd.DataFrame(bs)
        q = lambda c: tuple(np.nanquantile(c, [0.05, 0.95]))
        dm = bs["mean_rcm"] - bs["mean_gcm"]
        # resampling the years adds noise to the means, which lowers r in every replicate: the interval of r keeps the
        # spread of the replicates around their median, centred on r
        r_med = np.nanmedian(bs["r"])
        rows.append(dict(var=var, season=s, n_years=n, unit="%" if var == "pr" else "K", **st,
                         r_lo=st["r"] - (r_med - q(bs["r"])[0]), r_hi=min(1.0, st["r"] + (q(bs["r"])[1] - r_med)),
                         mean_gcm_lo=q(bs["mean_gcm"])[0], mean_gcm_hi=q(bs["mean_gcm"])[1],
                         mean_rcm_lo=q(bs["mean_rcm"])[0], mean_rcm_hi=q(bs["mean_rcm"])[1],
                         diff_lo=q(dm)[0], diff_hi=q(dm)[1],
                         share_sig_gcm=float((W * sig_g).sum()), share_sig_rcm=float((W * sig_r).sum()),
                         sign_agree_sig=agree, r_half_gcm=rh_g, r_half_rcm=rh_r, rel_gcm=rel_g, rel_rcm=rel_r,
                         r_max=r_max, r_over_rmax=st["r"] / r_max if np.isfinite(r_max) else np.nan))
        MAPS[var, s] = (dg, dr, sig_g, sig_r)
    print(f"{var}: done")

RES = pd.DataFrame(rows)
RES.to_csv(OUT / f"hosing_signal_rcm_vs_gcm_{TAG}.csv", index=False)
with pd.option_context("display.width", 250, "display.max_columns", None, "display.float_format", "{:.2f}".format):
    display(RES[["var", "season", "unit", "mean_gcm", "mean_gcm_lo", "mean_gcm_hi", "mean_rcm", "mean_rcm_lo", "mean_rcm_hi",
                 "diff_lo", "diff_hi", "r", "r_lo", "r_hi", "r_max", "r_over_rmax", "sd_ratio", "rmsd",
                 "share_sig_gcm", "share_sig_rcm", "sign_agree_sig"]])
print("saved:", OUT / f"hosing_signal_rcm_vs_gcm_{TAG}.csv")

### Summary per variable and season

Conventions of this summary (to discuss, not a test): pattern "consistent" if r / r_max >= 0.8, "partly" if
0.5-0.8, "different" below 0.5 (or r_max undefined: no reproducible change in one of the models); mean change
"different" if the 90 % interval of RCM - GCM excludes 0.

In [ ]:
def label(row):
    rr = row["r_over_rmax"]
    pat = ("no reproducible change" if not np.isfinite(rr) else
           "consistent" if rr >= 0.8 else "partly" if rr >= 0.5 else "different")
    mean = "different" if (row["diff_lo"] > 0 or row["diff_hi"] < 0) else "within noise"
    return pd.Series({"pattern": pat, "mean change": mean})


SUM = pd.concat([RES[["var", "season"]], RES.apply(label, axis=1),
                 RES[["mean_gcm", "mean_rcm", "unit", "r", "r_max", "sd_ratio"]].round(2)], axis=1)
SUM.to_csv(OUT / f"hosing_signal_summary_{TAG}.csv", index=False)
display(SUM)

## 4. Maps of the changes on the GCM grid

Per variable: rows = seasons; columns = EC-Earth3, RegCM5 0.25 aggregated onto the EC-Earth3 grid, RCM - GCM. Dots:
change significant at `ALPHA` (Welch t-test on the yearly seasonal means).

In [ ]:
def unflat(v):
    out = np.full(DOM.size, np.nan); out[IDX] = v
    return out.reshape(DOM.shape)


def lim(arrs, q=0.98):
    a = np.abs(np.concatenate([np.ravel(x)[np.isfinite(np.ravel(x))] for x in arrs]))
    return float(np.quantile(a, q)) if a.size else 1.0


def dots(ax, sig):
    yy, xx = np.nonzero(unflat(sig.astype(float)) == 1)
    ax.scatter(GLON[xx], GLAT[yy], s=0.6, c="k", lw=0)


for var in VARS:
    unit = "%" if var == "pr" else "K"
    cmap = "BrBG" if var == "pr" else "RdBu_r"
    seas = [s for s in SEASONS if s != "year"]
    v1 = lim([MAPS[var, s][k] for s in seas for k in (0, 1)])
    v2 = lim([MAPS[var, s][1] - MAPS[var, s][0] for s in seas])
    fig, axes = plt.subplots(len(seas), 3, figsize=(15, 3.4 * len(seas)), squeeze=False)
    for i, s in enumerate(seas):
        dg, dr, sg, sr = MAPS[var, s]
        r = RES[(RES["var"] == var) & (RES["season"] == s)].iloc[0]
        for j, (fld, vv, cm, ttl) in enumerate(((dg, v1, cmap, f"EC-Earth3, mean {r.mean_gcm:+.2f} {unit}"),
                                                 (dr, v1, cmap, f"RegCM5 0.25 on the GCM grid, mean {r.mean_rcm:+.2f} {unit}"),
                                                 (dr - dg, v2, "PuOr_r", f"RCM - GCM, r = {r.r:.2f} (r_max {r.r_max:.2f})"))):
            ax = axes[i, j]
            pm = ax.pcolormesh(GLON, GLAT, unflat(fld), shading="nearest", cmap=cm, vmin=-vv, vmax=vv)
            if j < 2:
                dots(ax, (sg, sr)[j])
            bu.draw_borders(ax, BORDERS, lw=0.3)
            ax.set_xlim(*REGION["lon"]); ax.set_ylim(*REGION["lat"])
            ax.set_title(f"{var} {s}: {ttl}", fontsize=9)
            plt.colorbar(pm, ax=ax, shrink=0.8, label=f"hosing - no hosing [{unit}]" if j < 2 else f"[{unit}]")
    fig.suptitle(f"{var}: hosing - no hosing, 2050 pair, {TAG}", y=1.0)
    plt.tight_layout(); plt.savefig(OUT / f"maps_{var}_{TAG}.png", dpi=110, bbox_inches="tight"); plt.show()

## 5. Cell by cell, and the correlations against the noise ceiling

In [ ]:
fig, axes = plt.subplots(len(VARS), 4, figsize=(16, 4 * len(VARS)), squeeze=False)
for i, var in enumerate(VARS):
    unit = "%" if var == "pr" else "K"
    for j, s in enumerate(("DJF", "MAM", "JJA", "SON")):
        dg, dr, _, _ = MAPS[var, s]
        ax = axes[i, j]
        ok = np.isfinite(dg) & np.isfinite(dr)
        ax.scatter(dg[ok], dr[ok], s=4, alpha=0.5)
        lo, hi = np.nanmin(np.r_[dg[ok], dr[ok]]), np.nanmax(np.r_[dg[ok], dr[ok]])
        ax.plot([lo, hi], [lo, hi], "k--", lw=0.8)
        r = RES[(RES["var"] == var) & (RES["season"] == s)].iloc[0]
        ax.set_title(f"{var} {s}: r = {r.r:.2f}, SD ratio {r.sd_ratio:.2f}", fontsize=9)
        ax.set_xlabel(f"EC-Earth3 [{unit}]"); ax.set_ylabel(f"RegCM5 0.25 [{unit}]")
plt.tight_layout(); plt.savefig(OUT / f"scatter_{TAG}.png", dpi=110); plt.show()

fig, ax = plt.subplots(figsize=(12, 4))
lab = [f"{v}\n{s}" for v, s in zip(RES["var"], RES["season"])]
x = np.arange(len(RES))
ax.errorbar(x, RES["r"], yerr=[RES["r"] - RES["r_lo"], RES["r_hi"] - RES["r"]], fmt="o", capsize=3,
            label="r (RCM vs GCM), 90 % bootstrap interval")
ax.plot(x, RES["r_max"], "_", ms=18, mew=2, c="C3", label="noise ceiling r_max")
ax.set_xticks(x); ax.set_xticklabels(lab, fontsize=8); ax.axhline(0, c="k", lw=0.5)
ax.set_ylabel("pattern correlation of the changes"); ax.legend(fontsize=8)
ax.set_title(f"Hosing - no hosing, 2050 pair, {TAG}: RegCM5 0.25 against EC-Earth3")
plt.tight_layout(); plt.savefig(OUT / f"correlations_{TAG}.png", dpi=110); plt.show()

## 6. The changes at 0.25 deg (RCM native grid) next to the GCM

What step 2 would carry into the product: the fine-scale structure of the RCM change (relaxation zone shaded).

In [ ]:
for var in VARS:
    unit = "%" if var == "pr" else "K"
    cmap = "BrBG" if var == "pr" else "RdBu_r"
    fig, axes = plt.subplots(2, 2, figsize=(14, 9), squeeze=False)
    for i, s in enumerate(("DJF", "JJA")):
        months = SEASONS[s]
        nat = change(season_means(RCM_NAT["hos", var], months).mean(0), season_means(RCM_NAT["nohos", var], months).mean(0), var)
        vv = lim([MAPS[var, s][0], MAPS[var, s][1]])
        ax = axes[i, 0]
        pm = ax.pcolormesh(GLON, GLAT, unflat(MAPS[var, s][0]), shading="nearest", cmap=cmap, vmin=-vv, vmax=vv)
        ax.set_title(f"{var} {s}: EC-Earth3 (domain of the comparison)", fontsize=9)
        ax = axes[i, 1]
        ax.pcolormesh(RLON, RLAT, np.where(INNER, nat, np.nan), shading="nearest", cmap=cmap, vmin=-vv, vmax=vv)
        ax.pcolormesh(RLON, RLAT, np.where(INNER, np.nan, nat), shading="nearest", cmap=cmap, vmin=-vv, vmax=vv, alpha=0.3)
        ax.set_title(f"{var} {s}: RegCM5 0.25, native grid", fontsize=9)
        for a in axes[i]:
            bu.draw_borders(a, BORDERS, lw=0.3); a.set_xlim(*REGION["lon"]); a.set_ylim(*REGION["lat"])
        plt.colorbar(pm, ax=axes[i].tolist(), shrink=0.8, label=f"hosing - no hosing [{unit}]")
    plt.savefig(OUT / f"maps_native_{var}_{TAG}.png", dpi=110, bbox_inches="tight"); plt.show()

## 7. Context: the RCM climate against its driver (no hosing)

Domain mean of RCM - GCM per season (K, or % of the GCM for pr), no-hosing run, `YEARS`: the distance between the
climates of the two models that step 2 would remove.

In [ ]:
rows2 = []
for var in VARS:
    for s, months in SEASONS.items():
        for run in ("nohos", "hos"):
            g = season_means(GCM[run, var], months).mean(0).ravel()[IDX]
            r = season_means(RCM[run, var], months).mean(0).ravel()[IDX]
            d = 100 * (r / np.where(g > 0, g, np.nan) - 1) if var == "pr" else r - g
            ok = np.isfinite(d)
            rows2.append(dict(var=var, season=s, run=run, unit="%" if var == "pr" else "K",
                              rcm_minus_gcm=float((W[ok] * d[ok]).sum() / W[ok].sum()),
                              p10=float(np.nanquantile(d, 0.1)), p90=float(np.nanquantile(d, 0.9))))
CTX = pd.DataFrame(rows2)
CTX.to_csv(OUT / f"rcm_minus_gcm_{TAG}.csv", index=False)
display(CTX.pivot_table(index=["var", "season"], columns="run", values="rcm_minus_gcm").round(2))